# 第13章 - 高级词嵌入技术

本notebook整合了以下内容:
- GloVe:全局向量词嵌入
- 子词嵌入(fastText和字节对编码)
- 词相似性和类比任务
- 预训练词向量的应用

## 1. GloVe模型

### 1.1 从Word2Vec到GloVe

**Word2Vec的视角**:
- Skip-gram使用softmax计算条件概率
- 目标:最小化交叉熵损失
- 问题:每次梯度计算需要遍历整个词表

**用全局统计量重新表示Skip-gram**:

定义:
- $\mathcal{C}_i$: 词$w_i$的所有上下文词的多重集合
- $x_{ij}$: 词$w_j$在$\mathcal{C}_i$中出现的次数(共现次数)
- $x_i = |\mathcal{C}_i|$: 所有上下文词总数
- $p_{ij} = x_{ij}/x_i$: 经验条件概率

Skip-gram的损失可以重写为:
$$-\sum_i x_i \sum_j p_{ij} \log q_{ij}$$

这正是$p_{ij}$和$q_{ij}$之间的交叉熵!

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import os

# 设置随机种子
torch.manual_seed(42)

### 1.2 GloVe的核心思想

**问题**:交叉熵损失的缺点
1. 计算代价高(需要归一化)
2. 对罕见事件权重过大

**解决方案**:使用平方损失

**关键洞察:共现比率**

考虑词对(ice, steam)和候选词(solid, gas, water, fashion):

| 词 $w_k$ | $P(w_k|\text{ice})$ | $P(w_k|\text{steam})$ | $P(w_k|\text{ice})/P(w_k|\text{steam})$ |
|---------|---------------------|----------------------|----------------------------------------|
| solid   | $1.9 \times 10^{-4}$ | $2.2 \times 10^{-5}$ | **8.9** (与ice相关) |
| gas     | $6.6 \times 10^{-5}$ | $7.8 \times 10^{-4}$ | **0.085** (与steam相关) |
| water   | $3.0 \times 10^{-3}$ | $2.2 \times 10^{-3}$ | 1.36 (两者都相关) |
| fashion | $1.7 \times 10^{-5}$ | $1.8 \times 10^{-5}$ | 0.96 (两者都不相关) |

**观察**:
- 比率远大于1: $w_k$更可能与ice相关
- 比率远小于1: $w_k$更可能与steam相关
- 比率接近1: $w_k$与两者都相关或都不相关

→ **共现比率比单独的概率更能捕捉词之间的关系!**

### 1.3 GloVe模型推导

**目标**:设计函数$f$使得
$$f(\mathbf{u}_j, \mathbf{u}_k, \mathbf{v}_i) \approx \frac{p_{ij}}{p_{ik}}$$

**推导步骤**:

1. **对称性要求**: $f$应该对$j$和$k$对称
   $$f(\mathbf{u}_j - \mathbf{u}_k, \mathbf{v}_i) \approx \frac{p_{ij}}{p_{ik}}$$

2. **加法性质**: 要求$f(\mathbf{x})f(-\mathbf{x})=1$
   → 唯一解: $f(\mathbf{x}) = \exp(\mathbf{x})$

3. **点积形式**:
   $$\exp(\mathbf{u}_j^\top \mathbf{v}_i) \approx \alpha p_{ij} = \alpha \frac{x_{ij}}{x_i}$$

4. **取对数并添加偏置**:
   $$\mathbf{u}_j^\top \mathbf{v}_i + b_i + c_j \approx \log x_{ij}$$
   
   其中$b_i = \log \alpha - \log x_i$, $c_j$是额外的偏置

5. **加权平方损失**:
   $$\sum_{i,j \in \mathcal{V}} h(x_{ij})(\mathbf{u}_j^\top \mathbf{v}_i + b_i + c_j - \log x_{ij})^2$$

**权重函数** $h(x)$:
- 当$x=0$时,$h(x)=0$(跳过非共现词对)
- 当$x$很大时,$h(x)$不应太大(避免过度加权高频词)
- 建议: $h(x) = \begin{cases} (x/c)^\alpha & \text{if } x < c \\ 1 & \text{otherwise} \end{cases}$
- 常用参数: $c=100$, $\alpha=0.75$

### 1.4 GloVe vs Word2Vec

| 特性 | Word2Vec (Skip-gram) | GloVe |
|------|---------------------|-------|
| 概率建模 | 条件概率$P(w_o|w_c)$ | 联合概率的对数$\log x_{ij}$ |
| 损失函数 | 交叉熵 | 加权平方损失 |
| 统计信息 | 局部(滑动窗口) | 全局(共现矩阵) |
| 对称性 | 非对称($v_i \neq u_i$) | 对称($v_i$和$u_i$等价) |
| 计算效率 | 需要多次迭代 | 预计算共现,更快 |
| 输出向量 | $\mathbf{v}_i$(或$\mathbf{u}_i$) | $\mathbf{v}_i + \mathbf{u}_i$ |

**GloVe的优势**:
1. 利用全局统计信息
2. 对罕见事件更稳健(加权函数)
3. 训练更快(预计算共现)
4. 效果通常优于或相当于Word2Vec

## 2. 子词嵌入

### 2.1 动机

**问题**:Word2Vec和GloVe的局限
- "help", "helps", "helped", "helping"被视为完全不同的词
- "dog"和"dogs"没有共享参数
- 罕见词和词表外(OOV)词表示质量差

**语言学观察**:
- 英语: 动词有多种变形(时态、人称)
- 法语/西班牙语: 动词可有40+变形
- 芬兰语: 名词可有15种格变形
- 德语: 复合词(Donaudampfschifffahrt = 多瑙河蒸汽船航运)

**解决方案**:子词(Subword)嵌入
- 将词分解为更小的单位
- 共享子词级别的参数
- 词向量 = 子词向量之和

### 2.2 fastText模型

**字符n-gram方法**:

对于词"where":
1. 添加边界标记: `<where>`
2. 提取所有长度为3-6的字符n-gram:
   - 3-gram: `<wh`, `whe`, `her`, `ere`, `re>`
   - 4-gram: `<whe`, `wher`, `here`, `ere>`
   - 5-gram: `<wher`, `where`, `here>`
   - 6-gram: `<where>`
3. 特殊子词(整个词): `<where>`

**数学表示**:

设$\mathcal{G}_w$为词$w$的所有子词集合,则:
$$\mathbf{v}_w = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g$$

其中$\mathbf{z}_g$是子词$g$的向量

**优势**:
- 罕见词也能获得好的表示(通过频繁子词)
- OOV词可以表示(只要子词在词表中)
- 捕捉形态学信息

**劣势**:
- 词表更大(所有可能的n-gram)
- 计算成本更高(求和操作)
- 可能学到无意义的子词

In [ ]:
# fastText子词提取示例
def get_subwords(word, min_n=3, max_n=6):
    """提取词的所有字符n-gram子词"""
    word = f'<{word}>'  # 添加边界标记
    subwords = []
    # 提取所有长度在[min_n, max_n]的n-gram
    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            subwords.append(word[i:i+n])
    # 添加整个词作为特殊子词
    subwords.append(word)
    return subwords

# 示例
examples = ['where', 'help', 'helping', 'unhelpful']
for word in examples:
    subwords = get_subwords(word)
    print(f'{word}: {len(subwords)} 个子词')
    print(f'  {subwords[:5]}...')  # 显示前5个
    print()

### 2.3 字节对编码(BPE)

**问题**:fastText的词表大小不可控
- 所有3-6字符的n-gram数量巨大
- 内存和计算开销大

**BPE的解决方案**:
- 数据驱动的子词发现
- 固定词表大小
- 可变长度的子词

**算法流程**:

1. **初始化**:词表 = 所有单字符 + 特殊符号
2. **迭代**:
   - 统计所有相邻符号对的频率
   - 合并最频繁的符号对
   - 将新符号加入词表
   - 重复直到达到目标词表大小
3. **分词**:贪心地匹配最长子词

In [ ]:
import collections

# 初始化符号词表
symbols = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
           'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z',
           '_', '[UNK]']

# 示例数据集(词频统计)
raw_token_freqs = {'fast_': 4, 'faster_': 3, 'tall_': 5, 'taller_': 4}

# 在字符之间插入空格
token_freqs = {}
for token, freq in raw_token_freqs.items():
    token_freqs[' '.join(list(token))] = freq

print('初始token_freqs:')
for token, freq in token_freqs.items():
    print(f'  {token}: {freq}')

In [ ]:
def get_max_freq_pair(token_freqs):
    """返回最频繁的相邻符号对"""
    pairs = collections.defaultdict(int)
    for token, freq in token_freqs.items():
        symbols = token.split()
        for i in range(len(symbols) - 1):
            # 统计相邻符号对的频率
            pairs[symbols[i], symbols[i + 1]] += freq
    return max(pairs, key=pairs.get)

def merge_symbols(max_freq_pair, token_freqs, symbols):
    """合并最频繁的符号对"""
    symbols.append(''.join(max_freq_pair))
    new_token_freqs = dict()
    for token, freq in token_freqs.items():
        # 替换所有出现的符号对
        new_token = token.replace(
            ' '.join(max_freq_pair), ''.join(max_freq_pair))
        new_token_freqs[new_token] = freq
    return new_token_freqs

# 执行10次合并
num_merges = 10
for i in range(num_merges):
    max_freq_pair = get_max_freq_pair(token_freqs)
    token_freqs = merge_symbols(max_freq_pair, token_freqs, symbols)
    print(f'合并#{i+1}: {max_freq_pair}')

print(f'\n最终符号表(新增的{num_merges}个):')
print(symbols[-num_merges:])

In [ ]:
print('\n最终分词结果:')
for token in token_freqs.keys():
    print(f'  {token}')

In [ ]:
# BPE分词函数
def segment_BPE(tokens, symbols):
    """使用BPE对新词进行分词"""
    outputs = []
    for token in tokens:
        start, end = 0, len(token)
        cur_output = []
        # 贪心匹配最长子词
        while start < len(token) and start < end:
            if token[start:end] in symbols:
                cur_output.append(token[start:end])
                start = end
                end = len(token)
            else:
                end -= 1
        if start < len(token):
            cur_output.append('[UNK]')
        outputs.append(' '.join(cur_output))
    return outputs

# 对新词进行分词
new_tokens = ['tallest_', 'fatter_']
print(f'新词BPE分词:')
for token, segmented in zip(new_tokens, segment_BPE(new_tokens, symbols)):
    print(f'  {token} → {segmented}')

**BPE的应用**:
- GPT-2: BPE with 50,257 tokens
- RoBERTa: Byte-level BPE
- BERT: WordPiece (BPE的变体)

**WordPiece vs BPE**:
- BPE: 选择频率最高的符号对
- WordPiece: 选择使似然增加最多的符号对
- 效果相似,WordPiece理论上更优

## 3. 词相似性和类比任务

### 3.1 加载预训练词向量

**GloVe预训练模型**:
- glove.6B.50d: 60亿tokens, 50维
- glove.6B.100d: 60亿tokens, 100维
- glove.42B.300d: 420亿tokens, 300维

**fastText预训练模型**:
- wiki.en: 维基百科, 300维

In [ ]:
# 注册下载链接
d2l.DATA_HUB['glove.6b.50d'] = (
    d2l.DATA_URL + 'glove.6B.50d.zip',
    '0b8703943ccdb6eb788e6f091b8946e82231bc4d')

d2l.DATA_HUB['glove.6b.100d'] = (
    d2l.DATA_URL + 'glove.6B.100d.zip',
    'cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a')

# TokenEmbedding类
class TokenEmbedding:
    """GloVe/fastText嵌入加载器"""
    def __init__(self, embedding_name):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(
            embedding_name)
        self.unknown_idx = 0
        self.token_to_idx = {
            token: idx for idx, token in enumerate(self.idx_to_token)}

    def _load_embedding(self, embedding_name):
        idx_to_token, idx_to_vec = ['<unk>'], []
        data_dir = d2l.download_extract(embedding_name)
        
        # 读取嵌入文件
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            for line in f:
                elems = line.rstrip().split(' ')
                token, elems = elems[0], [float(elem) for elem in elems[1:]]
                # 跳过标题行(fastText)
                if len(elems) > 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)
        
        # <unk>的向量设为全0
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec
        return idx_to_token, torch.tensor(idx_to_vec)

    def __getitem__(self, tokens):
        """获取token(s)的向量"""
        indices = [self.token_to_idx.get(token, self.unknown_idx)
                   for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs

    def __len__(self):
        return len(self.idx_to_token)

# 加载GloVe 50维
glove_6b50d = TokenEmbedding('glove.6b.50d')
print(f'词表大小: {len(glove_6b50d)}')

In [ ]:
# 查询词的索引
print(f'"beautiful"的索引: {glove_6b50d.token_to_idx["beautiful"]}')
print(f'索引3367的词: {glove_6b50d.idx_to_token[3367]}')

# 获取词向量
vec = glove_6b50d[['beautiful']]
print(f'"beautiful"的向量形状: {vec.shape}')
print(f'向量的前5维: {vec[0, :5]}')

### 3.2 词相似度

**余弦相似度**:
$$\text{cosine}(\mathbf{x}, \mathbf{y}) = \frac{\mathbf{x}^\top \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}$$

**K近邻算法**:找到余弦相似度最高的K个词

In [ ]:
def knn(W, x, k):
    """K近邻搜索"""
    # 计算余弦相似度
    cos = torch.mv(W, x.reshape(-1,)) / (
        torch.sqrt(torch.sum(W * W, axis=1) + 1e-9) *
        torch.sqrt((x * x).sum()))
    # 找到top-k
    _, topk = torch.topk(cos, k=k)
    return topk, [cos[int(i)] for i in topk]

def get_similar_tokens(query_token, k, embed):
    """找到最相似的k个词"""
    topk, cos = knn(embed.idx_to_vec, 
                    embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):  # 排除查询词本身
        print(f'{embed.idx_to_token[int(i)]}: '
              f'cosine相似度={float(c):.3f}')

# 测试
print('与"chip"最相似的词:')
get_similar_tokens('chip', 3, glove_6b50d)

print('\n与"baby"最相似的词:')
get_similar_tokens('baby', 3, glove_6b50d)

print('\n与"beautiful"最相似的词:')
get_similar_tokens('beautiful', 3, glove_6b50d)

### 3.3 词类比任务

**任务定义**:
- 给定:"man" : "woman" :: "son" : ?
- 求解:找到与"daughter"最接近的词

**数学表示**:
$$\mathbf{v}_a : \mathbf{v}_b :: \mathbf{v}_c : \mathbf{v}_d$$

**求解方法**:
$$\mathbf{v}_d \approx \mathbf{v}_c + (\mathbf{v}_b - \mathbf{v}_a)$$

即:找到与$\mathbf{v}_c + \mathbf{v}_b - \mathbf{v}_a$最接近的词

**经典例子**:
- king - man + woman ≈ queen
- Paris - France + Germany ≈ Berlin
- big - bigger + small ≈ smaller

In [ ]:
def get_analogy(token_a, token_b, token_c, embed):
    """词类比: a:b :: c:?"""
    vecs = embed[[token_a, token_b, token_c]]
    # 计算 c + (b - a)
    x = vecs[1] - vecs[0] + vecs[2]
    # 找到最接近的词
    topk, cos = knn(embed.idx_to_vec, x, 1)
    return embed.idx_to_token[int(topk[0])]

# 性别类比
print('性别类比:')
print(f'man:woman :: son:{get_analogy("man", "woman", "son", glove_6b50d)}')
print(f'man:woman :: king:{get_analogy("man", "woman", "king", glove_6b50d)}')

# 地理类比
print('\n地理类比:')
print(f'paris:france :: berlin:{get_analogy("paris", "france", "berlin", glove_6b50d)}')
print(f'beijing:china :: tokyo:{get_analogy("beijing", "china", "tokyo", glove_6b50d)}')

# 形容词比较级
print('\n比较级类比:')
print(f'big:bigger :: small:{get_analogy("big", "bigger", "small", glove_6b50d)}')
print(f'bad:worse :: good:{get_analogy("bad", "worse", "good", glove_6b50d)}')

## 4. 小结

### 4.1 GloVe模型

**核心思想**:
- 利用全局共现统计量
- 共现比率比单独概率更有信息量
- 加权平方损失代替交叉熵

**优势**:
- 训练更快(预计算共现矩阵)
- 对罕见事件更稳健
- 对称性(可同时使用$\mathbf{v}_i$和$\mathbf{u}_i$)

**公式回顾**:
$$\min \sum_{i,j} h(x_{ij})(\mathbf{u}_j^\top \mathbf{v}_i + b_i + c_j - \log x_{ij})^2$$

### 4.2 子词嵌入

**fastText**:
- 字符n-gram(3-6)
- 词向量 = 子词向量之和
- 优势:处理OOV,捕捉形态
- 劣势:词表大,计算慢

**BPE**:
- 数据驱动的子词发现
- 固定词表大小
- 可变长度子词
- 广泛用于现代NLP(GPT, BERT)

### 4.3 词向量评估

**内在评估**:
- 词相似度:人工标注vs余弦相似度
- 词类比:准确率

**外在评估**:
- 下游任务性能(分类、NER等)
- 更可靠但成本高

### 4.4 局限性

**所有静态词嵌入的共同问题**:
- 上下文无关:"bank"(银行/河岸)同一向量
- 无法处理一词多义
- 忽略句法结构

**解决方向**:
- ELMo: 基于BiLSTM的上下文表示
- GPT: Transformer decoder
- **BERT**: Transformer encoder(下一节!)

### 练习

1. 比较GloVe不同维度(50, 100, 300)的效果
2. 在自己的数据集上训练BPE并观察分词结果
3. 尝试更多词类比任务,分析失败案例
4. 研究fastText如何处理拼写错误的词
5. 实现WordPiece算法并与BPE对比